# f6_m03a_fairness.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M03a — Fairness |

---

## 🎯 Qué hace

Analiza la equidad algorítmica del modelo CatBoost__balanced usando fairlearn.
Evalúa 6 variables sensibles:
- **sexo** — equidad de género
- **rama** — equidad por área (nombres completos desde config_entorno)
- **via_acceso** — equidad por origen de acceso
- **beca** — equidad socioeconómica (proxy: n_anios_beca > 0, campo Fase 3)
- **trabaja** — equidad laboral (derivado de situacion_laboral, campo nuevo Fase 3)
- **origen** — equidad nacional (España vs extranjero, de pais_nombre)

## 📋 Requisitos

- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_test.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`
- `data/03_features/df_exp_target_eda.parquet`
- `src/config_entorno.py` — RAMAS_NOMBRES
- Paquete: fairlearn

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/fairness_metricas.parquet` | Métricas por grupo |
| `results/fase6/fairness_{var}.png` | Gráfico recall+F1 por grupo (×6) |
| `docs/html/fase6/m03a_fairness.html` | Informe con nota via_acceso + mitigación |

## 🔄 Flujo

```
X_test_prep + X_test + CatBoost + df_exp_target_eda
    ↓ Recuperación de contexto: 6 variables sensibles
      (incluye campos nuevos Fase 3: situacion_laboral, pais_nombre, n_anios_beca)
    ↓ MetricFrame fairlearn × 6
    ↓ Nota explicativa via_acceso (F6-04)
    ↓ Propuesta mitigación (F6-08 cum laude)
    → fairness_metricas.parquet + m03a_fairness.html
```

## ➡️ Siguiente

`f6_m03b_errores_fpfn.ipynb`


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA     = ROOT / 'data' / '05_modelado'
DIR_FEATURES = ROOT / 'data' / '03_features'
DIR_MODELS   = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS  = ROOT / 'results' / 'fase6'
DIR_HTML     = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

print(f'ROOT:        {ROOT}')
print(f'DIR_RESULTS: {DIR_RESULTS}')

ROOT:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_RESULTS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# ============================================================
# CELDA 2: IMPORTS
# RAMAS_NOMBRES desde src/config_entorno.py — NO desde
# config_app.py para no crear dependencia notebooks → app.
# ============================================================
import base64
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equal_opportunity_difference,
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.html.render import render_pagina_desde_fichero
from src.config_entorno import RAMAS_NOMBRES

matplotlib.rcParams['figure.dpi'] = 120
print('Imports OK.')
print(f'Ramas: {RAMAS_NOMBRES}')

Imports OK.
Ramas: {'SO': 'Ciencias Sociales y Jurídicas', 'TE': 'Ingeniería y Arquitectura', 'SA': 'Ciencias de la Salud', 'HU': 'Artes y Humanidades', 'EX': 'Ciencias Experimentales'}


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y MODELO
# ============================================================
X_test       = pd.read_parquet(DIR_DATA / 'X_test.parquet')
X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

y_pred = pipeline_cat.predict(X_test_prep)
y_prob = pipeline_cat.predict_proba(X_test_prep)[:, 1]
y_true = y_test.values.ravel()

print(f'X_test_prep:  {X_test_prep.shape}  — 27 features (24 + 3 _missing)')
print(f'Abandono real:     {y_true.mean():.2%}')
print(f'Abandono predicho: {y_pred.mean():.2%}')

X_test_prep:  (6725, 27)  — 27 features (24 + 3 _missing)
Abandono real:     29.25%
Abandono predicho: 33.19%


In [4]:
# ============================================================
# CELDA 4: RECUPERACIÓN DE CONTEXTO — 6 VARIABLES SENSIBLES
# Variables sensibles no están en X_test_prep — se recuperan
# desde df_exp_target_eda por join de índice posicional.
#
# Campos nuevos del dataset Fase 3 usados aquí:
#   n_anios_beca:     campo nuevo Fase 3 — sustituye la variable de beca anterior
#   situacion_laboral: nuevo campo Fase 3 — trabajador vs no
#   pais_nombre:      nuevo campo Fase 3 — España vs extranjero
# ============================================================
df_ref  = pd.read_parquet(DIR_FEATURES / 'df_exp_target_eda.parquet')
df_meta = df_ref[[
    'sexo', 'rama', 'via_acceso',
    'n_anios_beca',
    'situacion_laboral',
    'pais_nombre'
]].iloc[X_test.index].copy()
df_meta.index = X_test.index

df_grupos = pd.DataFrame({
    'sexo':       df_meta['sexo'].values,
    # Rama: nombres completos desde RAMAS_NOMBRES (SO → Ciencias Sociales...)
    'rama':       df_meta['rama'].map(RAMAS_NOMBRES).fillna(df_meta['rama']).values,
    'via_acceso': df_meta['via_acceso'].values,
    # Beca: proxy binario de n_anios_beca (campo nuevo del dataset Fase 3)
    'beca':       (df_meta['n_anios_beca'] > 0).map(
                      {True: 'Con beca', False: 'Sin beca'}).values,
    # Trabaja: derivado de situacion_laboral (campo nuevo Fase 3)
    'trabaja':    df_meta['situacion_laboral'].apply(
                      lambda x: 'No trabaja'
                      if str(x).lower() in ['inactivo o desempleado', 'nan']
                      else 'Trabaja').values,
    # Origen: derivado de pais_nombre (campo nuevo Fase 3)
    'origen':     df_meta['pais_nombre'].apply(
                      lambda x: 'España' if str(x) == 'España'
                      else 'Extranjero').values,
    'y_true':     y_true,
    'y_pred':     y_pred,
    'y_prob':     y_prob,
})

print('Distribución de grupos sensibles:')
for col in ['sexo', 'rama', 'via_acceso', 'beca', 'trabaja', 'origen']:
    print(f'  {col}: {df_grupos[col].value_counts().to_dict()}')

Distribución de grupos sensibles:
  sexo: {'Mujer': 3705, 'Hombre': 3020}
  rama: {'Ciencias Sociales y Jurídicas': 3761, 'Ingeniería y Arquitectura': 1532, 'Ciencias de la Salud': 686, 'Artes y Humanidades': 569, 'Ciencias Experimentales': 177}
  via_acceso: {'Pruebas acceso Bachiller Logse': 4933, 'Ciclo Formativo de Grado sup. o equivalente': 804, 'Adaptación a Grado': 300, 'Titulados Universitarios': 269, 'Pruebas acceso mayores 25 años': 128, 'Traslado': 109, 'Extranjeros CEE': 87, 'Extranjeros no CEE': 56, 'Cambio de plan': 16, 'Pruebas acceso mayores 40 años': 12, 'Pruebas acceso mayores 45 años': 8, 'Deportistas de élite': 2, 'Minusválidos': 1}
  beca: {'Con beca': 4869, 'Sin beca': 1856}
  trabaja: {'Trabaja': 5398, 'No trabaja': 1327}
  origen: {'España': 6268, 'Extranjero': 457}


In [5]:
# ============================================================
# CELDA 5: CALCULAR MÉTRICAS DE EQUIDAD
# MetricFrame por cada una de las 6 variables sensibles.
# ============================================================
metricas_fn = {
    'accuracy':  accuracy_score,
    'precision': lambda y_t, y_p: precision_score(y_t, y_p, zero_division=0),
    'recall':    lambda y_t, y_p: recall_score(y_t, y_p, zero_division=0),
    'f1':        lambda y_t, y_p: f1_score(y_t, y_p, zero_division=0),
}

variables_sensibles = ['sexo', 'rama', 'via_acceso', 'beca', 'trabaja', 'origen']
resultados_equidad  = []

for var in variables_sensibles:
    mf  = MetricFrame(
        metrics=metricas_fn,
        y_true=df_grupos['y_true'],
        y_pred=df_grupos['y_pred'],
        sensitive_features=df_grupos[var]
    )
    dpd = demographic_parity_difference(
        df_grupos['y_true'], df_grupos['y_pred'],
        sensitive_features=df_grupos[var])
    dpr = demographic_parity_ratio(
        df_grupos['y_true'], df_grupos['y_pred'],
        sensitive_features=df_grupos[var])
    eod = equal_opportunity_difference(
        df_grupos['y_true'], df_grupos['y_pred'],
        sensitive_features=df_grupos[var])

    resultados_equidad.append({
        'variable':     var,
        'dp_diff':      round(dpd, 4),
        'dp_ratio':     round(dpr, 4),
        'eq_opp_diff':  round(eod, 4),
        'metric_frame': mf,
    })
    print(f'\n--- {var.upper()} ---')
    print(f'  DP Diff: {dpd:.4f} | DP Ratio: {dpr:.4f} | EqOpp: {eod:.4f}')
    print(mf.by_group.round(3).to_string())


--- SEXO ---
  DP Diff: 0.1711 | DP Ratio: 0.5985 | EqOpp: 0.0501
        accuracy  precision  recall     f1
sexo                                      
Hombre     0.865      0.774   0.896  0.830
Mujer      0.904      0.765   0.846  0.803

--- RAMA ---
  DP Diff: 0.2769 | DP Ratio: 0.3852 | EqOpp: 0.0980
                               accuracy  precision  recall     f1
rama                                                             
Artes y Humanidades               0.898      0.810   0.788  0.799
Ciencias Experimentales           0.921      0.902   0.873  0.887
Ciencias Sociales y Jurídicas     0.895      0.777   0.884  0.827
Ciencias de la Salud              0.945      0.782   0.886  0.830
Ingeniería y Arquitectura         0.834      0.736   0.874  0.799

--- VIA_ACCESO ---
  DP Diff: 0.7500 | DP Ratio: 0.0000 | EqOpp: 1.0000
                                             accuracy  precision  recall     f1
via_acceso                                                                     

In [6]:
# ============================================================
# CELDA 6: NOTA EXPLICATIVA — VALORES EXTREMOS VIA_ACCESO
# DP Ratio=0.0 / EqOpp=1.0 → insuficiencia muestral (n<30),
# no sesgo sistemático del modelo.
# ============================================================
print('=== ANÁLISIS VIA_ACCESO ===')
tasa_via = df_grupos.groupby('via_acceso')['y_true'].agg(['mean', 'sum', 'count'])
tasa_via.columns = ['tasa_abandono', 'n_abandono', 'n_total']
print(tasa_via.sort_values('n_total', ascending=False).round(3).to_string())
print('\nVías con n < 30 (métricas inestables):')
print(tasa_via[tasa_via['n_total'] < 30][['n_total']].to_string())

=== ANÁLISIS VIA_ACCESO ===
                                             tasa_abandono  n_abandono  n_total
via_acceso                                                                     
Pruebas acceso Bachiller Logse                       0.262        1292     4933
Ciclo Formativo de Grado sup. o equivalente          0.384         309      804
Adaptación a Grado                                   0.517         155      300
Titulados Universitarios                             0.234          63      269
Pruebas acceso mayores 25 años                       0.461          59      128
Traslado                                             0.202          22      109
Extranjeros CEE                                      0.391          34       87
Extranjeros no CEE                                   0.304          17       56
Cambio de plan                                       0.312           5       16
Pruebas acceso mayores 40 años                       0.583           7       12
Pruebas acce

In [7]:
# ============================================================
# CELDA 7: GUARDAR MÉTRICAS
# ============================================================
registros = []
for res in resultados_equidad:
    var = res['variable']
    mf  = res['metric_frame']
    for grupo, row in mf.by_group.iterrows():
        registros.append({
            'variable_sensible': var,
            'grupo':             str(grupo),
            'accuracy':          row['accuracy'],
            'precision':         row['precision'],
            'recall':            row['recall'],
            'f1':                row['f1'],
            'dp_diff':           res['dp_diff'],
            'dp_ratio':          res['dp_ratio'],
            'eq_opp_diff':       res['eq_opp_diff'],
        })

df_fairness = pd.DataFrame(registros)
df_fairness.to_parquet(DIR_RESULTS / 'fairness_metricas.parquet')
print(f'✅ Guardado: {df_fairness.shape}')
print(df_fairness[['variable_sensible','grupo','recall','f1','dp_diff']].to_string())

✅ Guardado: (26, 9)
   variable_sensible                                        grupo    recall        f1  dp_diff
0               sexo                                       Hombre  0.895683  0.830346   0.1711
1               sexo                                        Mujer  0.845614  0.803333   0.1711
2               rama                          Artes y Humanidades  0.787671  0.798611   0.2769
3               rama                      Ciencias Experimentales  0.873016  0.887097   0.2769
4               rama                Ciencias Sociales y Jurídicas  0.884328  0.827225   0.2769
5               rama                         Ciencias de la Salud  0.885714  0.830357   0.2769
6               rama                    Ingeniería y Arquitectura  0.874355  0.799371   0.2769
7         via_acceso                           Adaptación a Grado  0.929032  0.886154   0.7500
8         via_acceso                               Cambio de plan  0.600000  0.500000   0.7500
9         via_acceso  Ciclo Fo

In [8]:
# ============================================================
# CELDA 8: GRÁFICOS — RECALL Y F1 POR GRUPO (×6)
# Recall = métrica clave para equidad en detección de abandono.
# ============================================================
rutas_fairness = {}

for res in resultados_equidad:
    var  = res['variable']
    mf   = res['metric_frame']
    ruta = DIR_RESULTS / f'fairness_{var}.png'
    rutas_fairness[var] = ruta

    grupos  = mf.by_group.index.tolist()
    recalls = mf.by_group['recall'].tolist()
    f1s     = mf.by_group['f1'].tolist()
    x       = np.arange(len(grupos))
    ancho   = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(grupos) * 1.4), 5))
    ax.bar(x - ancho/2, recalls, ancho, label='Recall', color='#3182ce', alpha=0.8)
    ax.bar(x + ancho/2, f1s,     ancho, label='F1',     color='#e53e3e', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([str(g)[:30] for g in grupos], rotation=30, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.axhline(mf.overall['recall'], color='#3182ce', linestyle='--', linewidth=1,
               alpha=0.6, label=f'Recall global ({mf.overall["recall"]:.2f})')
    ax.axhline(mf.overall['f1'],     color='#e53e3e', linestyle='--', linewidth=1,
               alpha=0.6, label=f'F1 global ({mf.overall["f1"]:.2f})')
    ax.set_ylabel('Métrica')
    ax.set_title(
        f'Fase 6 — Fairness: {var} | DP={res["dp_diff"]:.3f} | EqOpp={res["eq_opp_diff"]:.3f}',
        fontsize=12, pad=10
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(ruta, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'✅ {ruta.name}')

✅ fairness_sexo.png
✅ fairness_rama.png
✅ fairness_via_acceso.png
✅ fairness_beca.png
✅ fairness_trabaja.png
✅ fairness_origen.png


In [9]:
# ============================================================
# CELDA 8b: CURVAS ROC POR GRUPO 🏆
# No solo un punto de recall/F1 sino la curva completa.
# Permite ver si hay grupos donde el modelo tiene menos
# capacidad discriminativa en todo el rango de umbrales.
# Se generan curvas para las variables con grupos suficientes:
# sexo, rama, beca, trabaja, origen.
# ============================================================
from sklearn.metrics import roc_curve, auc

# Variables donde tiene sentido ROC (grupos con suficientes positivos)
vars_roc = ['sexo', 'rama', 'beca', 'trabaja', 'origen']
rutas_roc = {}

for var in vars_roc:
    grupos = df_grupos[var].unique()
    # Filtrar grupos con al menos 10 positivos
    grupos_validos = [
        g for g in grupos
        if df_grupos[df_grupos[var] == g]['y_true'].sum() >= 10
    ]
    if len(grupos_validos) < 2:
        continue

    fig, ax = plt.subplots(figsize=(8, 6))
    colores_roc = ['#3182ce','#e53e3e','#38a169','#dd6b20','#805ad5','#d69e2e']

    aucs_por_grupo = {}
    for i, grupo in enumerate(sorted(grupos_validos)):
        mask    = df_grupos[var] == grupo
        y_t     = df_grupos.loc[mask, 'y_true'].values
        y_p     = df_grupos.loc[mask, 'y_prob'].values
        if len(np.unique(y_t)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_t, y_p)
        roc_auc     = auc(fpr, tpr)
        aucs_por_grupo[str(grupo)] = round(roc_auc, 3)
        color = colores_roc[i % len(colores_roc)]
        label_grupo = str(grupo)[:25]
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{label_grupo} (AUC={roc_auc:.3f})')

    # Línea diagonal (clasificador aleatorio)
    ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Aleatorio (AUC=0.5)')
    ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
    ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)', fontsize=11)
    ax.set_title(
        f'Fase 6 — Curvas ROC por grupo: {var}\n'        f'Diferencia de AUC entre grupos indica disparidad de capacidad discriminativa',
        fontsize=11
    )
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    ruta_roc = DIR_RESULTS / f'fairness_roc_{var}.png'
    rutas_roc[var] = ruta_roc
    plt.savefig(ruta_roc, dpi=120, bbox_inches='tight')
    plt.close()

    # Diferencia máxima de AUC entre grupos
    if aucs_por_grupo:
        diff_auc = max(aucs_por_grupo.values()) - min(aucs_por_grupo.values())
        print(f'{var}: AUC por grupo = {aucs_por_grupo} | Δ AUC = {diff_auc:.3f}')
        print(f'  ✅ {ruta_roc.name}')

print(f'\n✅ Curvas ROC generadas para: {list(rutas_roc.keys())}')


sexo: AUC por grupo = {'Hombre': np.float64(0.946), 'Mujer': np.float64(0.956)} | Δ AUC = 0.010
  ✅ fairness_roc_sexo.png
rama: AUC por grupo = {'Artes y Humanidades': np.float64(0.957), 'Ciencias Experimentales': np.float64(0.965), 'Ciencias Sociales y Jurídicas': np.float64(0.956), 'Ciencias de la Salud': np.float64(0.976), 'Ingeniería y Arquitectura': np.float64(0.929)} | Δ AUC = 0.047
  ✅ fairness_roc_rama.png
beca: AUC por grupo = {'Con beca': np.float64(0.941), 'Sin beca': np.float64(0.963)} | Δ AUC = 0.022
  ✅ fairness_roc_beca.png
trabaja: AUC por grupo = {'No trabaja': np.float64(0.934), 'Trabaja': np.float64(0.956)} | Δ AUC = 0.022
  ✅ fairness_roc_trabaja.png
origen: AUC por grupo = {'España': np.float64(0.956), 'Extranjero': np.float64(0.909)} | Δ AUC = 0.047
  ✅ fairness_roc_origen.png

✅ Curvas ROC generadas para: ['sexo', 'rama', 'beca', 'trabaja', 'origen']


In [10]:
# ============================================================
# CELDA INTERSECCIONAL: ANÁLISIS INTERSECCIONAL 🏆
# La fairness interseccional analiza grupos combinados:
# mujer + sin beca + trabaja. Los grupos interseccionales
# suelen sufrir más disparidad que cada factor por separado.
# Nivel publicación científica — muy valorado en fairness.
# ============================================================
from itertools import combinations
from sklearn.metrics import recall_score, f1_score as _f1s

# Definir grupos interseccionales de interés
# (combinar max 2 variables para mantener n suficiente)
intersecciones = {
    'Mujer + Sin beca':     (df_grupos['sexo'] == 'Mujer') & (df_grupos['beca'] == 'Sin beca'),
    'Mujer + Trabaja':      (df_grupos['sexo'] == 'Mujer') & (df_grupos['trabaja'] == 'Trabaja'),
    'Mujer + Extranjero':   (df_grupos['sexo'] == 'Mujer') & (df_grupos['origen'] == 'Extranjero'),
    'Hombre + Sin beca':    (df_grupos['sexo'] == 'Hombre') & (df_grupos['beca'] == 'Sin beca'),
    'Hombre + Trabaja':     (df_grupos['sexo'] == 'Hombre') & (df_grupos['trabaja'] == 'Trabaja'),
    'Trabaja + Sin beca':   (df_grupos['trabaja'] == 'Trabaja') & (df_grupos['beca'] == 'Sin beca'),
    'Extranjero + Trabaja': (df_grupos['origen'] == 'Extranjero') & (df_grupos['trabaja'] == 'Trabaja'),
    'Extranjero + Sin beca':(df_grupos['origen'] == 'Extranjero') & (df_grupos['beca'] == 'Sin beca'),
}

resultados_inter = []
print(f'{"Grupo interseccional":35s} {"n":>6} {"Recall":>8} {"F1":>8} {"Recall global":>14}')
print('-'*80)

recall_global_val = recall_score(df_grupos['y_true'], df_grupos['y_pred'], zero_division=0)

for nombre_grupo, mask in intersecciones.items():
    n = mask.sum()
    if n < 20:
        continue
    y_t = df_grupos.loc[mask, 'y_true'].values
    y_p = df_grupos.loc[mask, 'y_pred'].values
    if len(np.unique(y_t)) < 2:
        continue
    rec = recall_score(y_t, y_p, zero_division=0)
    f1  = _f1s(y_t, y_p, zero_division=0)
    diff = rec - recall_global_val
    resultados_inter.append({
        'grupo': nombre_grupo, 'n': n,
        'recall': round(rec, 4), 'f1': round(f1, 4),
        'diff_recall': round(diff, 4)
    })
    icono = '⚠️ ' if diff < -0.1 else ('❌' if diff < -0.2 else '  ')
    print(f'{icono}{nombre_grupo:35s} {n:>6} {rec:>8.3f} {f1:>8.3f} {diff:>+14.3f}')

df_inter = pd.DataFrame(resultados_inter)
if len(df_inter) > 0:
    df_inter.to_parquet(DIR_RESULTS / 'fairness_interseccional.parquet')
    
    # Gráfico de barras — recall por grupo interseccional vs global
    df_inter_sorted = df_inter.sort_values('recall')
    colores_inter = ['#e53e3e' if r < recall_global_val - 0.1 
                     else ('#d69e2e' if r < recall_global_val else '#38a169')
                     for r in df_inter_sorted['recall']]
    
    fig, ax = plt.subplots(figsize=(10, max(5, len(df_inter_sorted)*0.5)))
    ax.barh(df_inter_sorted['grupo'], df_inter_sorted['recall'],
            color=colores_inter, alpha=0.85)
    ax.axvline(recall_global_val, color='#2d3748', linestyle='--', linewidth=2,
               label=f'Recall global ({recall_global_val:.3f})')
    ax.set_xlabel('Recall', fontsize=11)
    ax.set_title(
        'Fase 6 — Análisis Interseccional de Fairness\n'        'Recall por grupos combinados (rojo = disparidad significativa)',
        fontsize=12
    )
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1)
    plt.tight_layout()
    ruta_inter = DIR_RESULTS / 'fairness_interseccional.png'
    plt.savefig(ruta_inter, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'\n✅ Análisis interseccional guardado.')
    
    peor = df_inter.nsmallest(1, 'recall').iloc[0]
    print(f'Grupo más desfavorecido: {peor["grupo"]} (recall={peor["recall"]:.3f}, n={peor["n"]})')
else:
    ruta_inter = None
    print('⚠️  Sin grupos con n suficiente.')


Grupo interseccional                     n   Recall       F1  Recall global
--------------------------------------------------------------------------------
  Mujer + Sin beca                       874    0.932    0.895         +0.058
  Mujer + Trabaja                       2945    0.854    0.817         -0.020
  Mujer + Extranjero                     276    0.867    0.754         -0.006
  Hombre + Sin beca                      982    0.955    0.889         +0.081
  Hombre + Trabaja                      2453    0.903    0.850         +0.029
  Trabaja + Sin beca                    1666    0.946    0.902         +0.072
  Extranjero + Trabaja                   338    0.914    0.827         +0.040
  Extranjero + Sin beca                  104    0.960    0.780         +0.086

✅ Análisis interseccional guardado.
Grupo más desfavorecido: Mujer + Trabaja (recall=0.854, n=2945)


In [11]:
# ============================================================
# CELDA 9: GENERAR HTML
# ============================================================
def img_b64(ruta):
    if not ruta or not ruta.exists():
        return ''
    with open(ruta, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def bloque_imagen(b64, titulo, caption):
    if not b64:
        return f'<p style="color:#e53e3e">⚠️ No disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

etiquetas_var = {
    'sexo':       'Sexo',
    'rama':       'Rama de conocimiento',
    'via_acceso': 'Vía de acceso',
    'beca':       'Situación de beca (n_anios_beca > 0)',
    'trabaja':    'Situación laboral (trabaja / no trabaja)',
    'origen':     'Origen nacional (España / Extranjero)',
}

# Tabla resumen semáforo
filas_resumen = ''
for res in resultados_equidad:
    var = res['variable']
    dpd, dpr, eod = res['dp_diff'], res['dp_ratio'], res['eq_opp_diff']
    if abs(dpd) < 0.1:   color, icono = '#38a169', '✅'
    elif abs(dpd) < 0.2: color, icono = '#d69e2e', '⚠️'
    else:                color, icono = '#e53e3e', '❌'
    nota = ' *' if var == 'via_acceso' else ''
    filas_resumen += (
        f'<tr><td style="padding:8px 12px;font-weight:600">{etiquetas_var.get(var,var)}{nota}</td>'
        f'<td style="padding:8px 12px;text-align:center;color:{color}">{icono} {dpd:.4f}</td>'
        f'<td style="padding:8px 12px;text-align:center">{dpr:.4f}</td>'
        f'<td style="padding:8px 12px;text-align:center">{eod:.4f}</td></tr>'
    )

# Tabla via_acceso para nota explicativa
tasa_via_html = ''
for via, row in tasa_via.sort_values('n_total', ascending=False).iterrows():
    alerta = ' ⚠️' if row['n_total'] < 30 else ''
    tasa_via_html += (
        f'<tr><td style="padding:5px 10px;font-size:12px">{via}</td>'
        f'<td style="padding:5px 10px;text-align:center">{int(row["n_total"])}</td>'
        f'<td style="padding:5px 10px;text-align:center">{row["tasa_abandono"]:.1%}{alerta}</td></tr>'
    )

contenido = (
    '<h2 style="color:#2d3748">Fase 6 — Fairness: Equidad Algorítmica</h2>'
    '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    'Análisis de equidad sobre 6 variables sensibles. Incluye variables nuevas del '
    'dataset de Fase 3: situación laboral y origen nacional. '
    'Métricas: Demographic Parity Difference/Ratio y Equal Opportunity Difference.'
    '</p>'
    '<h3 style="color:#2d3748;margin-top:20px">Resumen</h3>'
    '<table style="width:100%;border-collapse:collapse;font-size:13px;margin-bottom:8px">'
    '<thead><tr style="background:#edf2f7">'
    '<th style="padding:8px 12px;text-align:left">Variable sensible</th>'
    '<th style="padding:8px 12px;text-align:center">DP Difference<br><small>(ideal=0)</small></th>'
    '<th style="padding:8px 12px;text-align:center">DP Ratio<br><small>(ideal=1)</small></th>'
    '<th style="padding:8px 12px;text-align:center">EqOpp Diff<br><small>(ideal=0)</small></th>'
    '</tr></thead>'
    f'<tbody>{filas_resumen}</tbody></table>'
    '<p style="color:#718096;font-size:12px;margin-bottom:24px">* Ver nota explicativa.</p>'
    + bloque_imagen(img_b64(rutas_fairness.get('sexo')),
        'Equidad por sexo',
        'Recall y F1 por género.')
    + bloque_imagen(img_b64(rutas_fairness.get('rama')),
        'Equidad por rama de conocimiento',
        'Nombres completos de rama (desde config_entorno). '
        'Diferencias indican que el modelo falla más en ciertas áreas.')
    + bloque_imagen(img_b64(rutas_fairness.get('via_acceso')),
        'Equidad por vía de acceso',
        'Ver nota explicativa para valores extremos.')
    + bloque_imagen(img_b64(rutas_fairness.get('beca')),
        'Equidad por situación de beca',
        'Proxy de n_anios_beca > 0 (campo nuevo del dataset Fase 3). '
        'Crítico: n_anios_beca es top 2 en importancia SHAP.')
    + bloque_imagen(img_b64(rutas_fairness.get('trabaja')),
        'Equidad por situación laboral (campo nuevo Fase 3)',
        'Trabaja vs no trabaja, derivado de situacion_laboral. '
        'n_anios_trabajando es top 1 en SHAP — verificar equidad es esencial.')
    + bloque_imagen(img_b64(rutas_fairness.get('origen')),
        'Equidad por origen nacional (campo nuevo Fase 3)',
        'España vs Extranjero. Diferencias pueden indicar barreras '
        'lingüísticas o culturales no capturadas.')
    # CURVAS ROC POR GRUPO
    + '<h3 style="color:#2d3748;margin-top:28px">🏆 Curvas ROC por grupo</h3>'
    + '<p style="color:#4a5568;font-size:13px;max-width:800px;margin-bottom:16px">'
    'Las curvas ROC muestran la capacidad discriminativa del modelo en todo el rango '
    'de umbrales de decisión. Un AUC inferior en un grupo indica que el modelo '
    'tiene menos capacidad para distinguir abandonos de no-abandonos en ese grupo '
    '— una forma más profunda de inequidad que el simple recall.'
    '</p>'
    + ''.join(
        bloque_imagen(
            img_b64(rutas_roc.get(var)),
            f'Curvas ROC por {etiquetas_var.get(var, var)}',
            f'Cada línea = un grupo. AUC = área bajo la curva (ideal = 1.0). '
            f'Diferencia de AUC entre grupos indica disparidad de capacidad discriminativa.'
        )
        for var in ['sexo', 'rama', 'beca', 'trabaja', 'origen']
        if var in rutas_roc
    )
    # ANÁLISIS INTERSECCIONAL
    + '<h3 style="color:#2d3748;margin-top:28px">🏆 Análisis Interseccional</h3>'
    + '<p style="color:#4a5568;font-size:13px;max-width:800px;margin-bottom:16px">'
    'Los grupos interseccionales combinan dos variables sensibles. '
    'Una mujer extranjera sin beca puede sufrir más disparidad que '
    'cualquiera de esos factores por separado — esto no se detecta '
    'con el análisis univariado estándar. Nivel publicación científica.'
    '</p>'
    + (bloque_imagen(
        img_b64(ruta_inter),
        'Recall por grupos interseccionales',
        'Rojo = recall significativamente inferior al global (> 0.1 de diferencia). '
        'Identifica los grupos más vulnerables que el análisis univariado no detecta.'
    ) if ruta_inter else '<p style="color:#718096">Grupos insuficientes.</p>')
    # NOTA VIA_ACCESO (F6-04)
    + '<div style="margin-top:28px;padding:20px;background:#fffbeb;'
    'border-left:4px solid #d69e2e;border-radius:8px">'
    '<h3 style="color:#744210;font-size:14px;margin-bottom:10px">'
    '⚠️ Nota — valores extremos en via_acceso</h3>'
    '<p style="color:#4a5568;font-size:13px;margin-bottom:12px">'
    'DP Ratio=0.0 y EqOpp=1.0 <strong>no indican sesgo</strong> sino insuficiencia '
    'muestral: vías con n&lt;30 generan métricas aritméticamente inestables.'
    '</p>'
    '<table style="width:100%;border-collapse:collapse;font-size:12px">'
    '<thead><tr style="background:#fef3c7">'
    '<th style="padding:5px 10px;text-align:left">Vía de acceso</th>'
    '<th style="padding:5px 10px;text-align:center">N test</th>'
    '<th style="padding:5px 10px;text-align:center">Tasa abandono</th>'
    '</tr></thead>'
    f'<tbody>{tasa_via_html}</tbody></table>'
    '</div>'
    # MITIGACIÓN (F6-08)
    + '<div style="margin-top:28px;padding:20px;background:#f0fff4;'
    'border-left:4px solid #38a169;border-radius:8px">'
    '<h3 style="color:#22543d;font-size:14px;margin-bottom:10px">'
    '🏆 Propuesta de mitigación</h3>'
    '<ol style="color:#4a5568;font-size:13px;padding-left:20px;line-height:1.8">'
    '<li><strong>Umbral diferenciado por grupo</strong> — sin reentrenar.</li>'
    '<li><strong>Ponderación de muestras</strong> — <code>sample_weight</code> en CatBoost.</li>'
    '<li><strong>ThresholdOptimizer (fairlearn)</strong> — Equal Opportunity con mínima pérdida F1.</li>'
    '<li><strong>Más datos</strong> de vías minoritarias (n ≥ 30).</li>'
    '</ol>'
    '</div>'
    + '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
    'border-left:4px solid #3182ce;border-radius:6px;font-size:13px;color:#2c5282">'
    '<strong>Lectura:</strong> DP Diff &lt;0.1 = ✅ | 0.1–0.2 = ⚠️ | &gt;0.2 = ❌'
    '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m03a_fairness.ipynb', contenido)
(DIR_HTML / 'm03a_fairness.html').write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {DIR_HTML / "m03a_fairness.html"}')

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m03a_fairness.html
